In [8]:
!pip install -q transformers accelerate bitsandbytes pandas pyarrow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.7 MB/s eta 0:00:00


In [9]:
!pip install -q huggingface_hub

from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

In [15]:
import os, sys, torch, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm

# If eval.metrics is not available in Colab, comment this out for now
# from eval.metrics import evaluate_dataframe

PROCESSED_DIR = "/content"
RESULTS_DIR = "/content/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [46]:
df = pd.read_parquet("/content/val.parquet")
print(df.head())
print(f"Evaluating {len(df):,} samples")

        id   source                                              title  \
0   476134  eclipse  [OS/2] Fix plugin code to allow building with ...   
1  1899867  gitbugs  Reflect BitrateMode::Constant to rc_max_rate a...   
2   497583  eclipse  Don't duplicate intl.* and signon.* prefs in a...   
3   463940  eclipse                     Contextual alternates not used   
4   330594  eclipse  mouse and status bar flicker quickly when movi...   

                                                body priority      team  \
0                                                          P3  platform   
1  The [`rc_min_rate`](https://searchfox.org/mozi...       P0   unknown   
2                                                          P3  frontend   
3                                                          P3  platform   
4                                                          P3  platform   

  raw_severity resolution  resolution_time_days  
0       normal      fixed                   NaN  
1   

In [47]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.truncation_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    quantization_config=bnb_config,
    device_map="auto",
)

model.eval()
newline_token = tokenizer.encode("\n", add_special_tokens=False)[0]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [48]:
FEW_SHOT_EXAMPLES = [
    {
        "title"   : "Assign a name to the error class _LEGACY_ERROR_TEMP_2024",
        "body"    : "Choose a proper name for the error class _LEGACY_ERROR_TEMP_2024 defined in core/src/main/resources/error/error-classes.json. The name should be short but complete.",
        "priority": "P3",
        "team"    : "platform",
    },
    {
        "title"   : "Use spark.sql.timestampType for data source inference",
        "body"    : "With the configuration spark.sql.timestampType, TIMESTAMP in Spark is a user-specified alias associated with one of the TIMESTAMP_LTZ and TIMESTAMP_NTZ variations. This is quite complicated to handle in data source inference.",
        "priority": "P2",
        "team"    : "database",
    },
    {
        "title"   : "Intermittent test failure: execution-timing async script ordering",
        "body"    : "Filed by: smolnar. Parsed log: treeherder. expected property 0 to be external script #2 but got external script #1. Intermittent failure on mozilla-beta.",
        "priority": "P4",
        "team"    : "frontend",
    },
    {
        "title"   : "Over-aggressive warning for ClientWaitSync must return TIMEOUT_EXPIRED",
        "body"    : "The warning is accurate but likely common for apps to query one or a few times before returning to the event loop. Rather than asking them to refactor or workaround, we should adjust the warning threshold.",
        "priority": "P0",
        "team"    : "backend",
    },
    {
        "title"   : "Assertion failure: exn.isObject() at WasmInstance.cpp:4067",
        "body"    : "The attached testcase crashes on mozilla-central revision 20250124. Build with debug, run with --fuzzing-safe --cpu-count=2 --ion-offthread-compile=off. Backtrace: MarkPendingExceptionAsTrap.",
        "priority": "P0",
        "team"    : "infra",
    },
    {
        "title"   : "cqlsh should prefer newer TLS version by default",
        "body"    : "Some new JDK releases started to disable TLSv1.0 and TLSv1.1. The current default in cqlsh should be updated to prefer TLSv1.2 or higher to maintain compatibility.",
        "priority": "P3",
        "team"    : "security",
    },
    {
        "title"   : "Startup crash running android sw-wr in debug build",
        "body"    : "webrender::device::gl::Device::new closure gl.rs:1417. ErrorReactingGl::get_integer_v crash on startup when running android sw-wr in debug build.",
        "priority": "P1",
        "team"    : "mobile",
    },
]

PROMPT_TEMPLATE = """\
### Incident report:
{title}
{body}
### Triage (always output severity P0-P4 and one team only):
"""

COMPLETION_TEMPLATE = "severity:{priority} | team:{team}\n"


In [49]:
INSTRUCTION = """\
You are an incident triage classifier.

Use the examples below to classify the final incident.

Allowed severities: P0, P1, P2, P3, P4
Allowed teams: mobile, security, infra, backend, frontend, database, platform, unknown

Return exactly one line and nothing else:
severity:<P0-P4> | team:<mobile|security|infra|backend|frontend|database|platform|unknown>

"""

In [50]:

def build_few_shot_prefix() -> str:
    prefix = ""
    for ex in FEW_SHOT_EXAMPLES:
        body   = ex["body"].strip()
        prompt = PROMPT_TEMPLATE.format(title=ex["title"], body="\n" + body)
        completion = COMPLETION_TEMPLATE.format(priority=ex["priority"], team=ex["team"])
        prefix += prompt + completion + "\n\n"
    return prefix


FEW_SHOT_PREFIX = build_few_shot_prefix()
# print(FEW_SHOT_PREFIX)

In [51]:
import re

VALID_TEAMS = [
    "mobile", "security", "infra", "backend",
    "frontend", "database", "platform", "unknown"
]

def clean_prediction(text):
    first_line = text.strip().split("\n")[0].strip()

    sev_match = re.search(r"P[0-4]", first_line)
    team_match = re.search(
        r"\b(mobile|security|infra|backend|frontend|database|platform|unknown)\b",
        first_line.lower()
    )

    severity = sev_match.group(0) if sev_match else "P2"
    team = team_match.group(0) if team_match else "unknown"

    return f"severity:{severity} | team:{team}"

In [53]:
predictions = []

with torch.inference_mode():
    for _, row in tqdm(df.iterrows(), total=len(df)):
        body = row["body"].strip() if row["body"] else ""

        prompt = INSTRUCTION + "\n" + FEW_SHOT_PREFIX + PROMPT_TEMPLATE.format(
        title=row["title"].strip(),
        body=("\n" + body) if body else "",
        )
        # print("prompt", prompt," ##ended")
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048,
        ).to(model.device)

        output = model.generate(
            **inputs,
            max_new_tokens=15,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=[tokenizer.eos_token_id, newline_token],
        )

        generated = output[0][inputs["input_ids"].shape[1]:]
        raw_text = tokenizer.decode(generated, skip_special_tokens=True)
        pred = clean_prediction(raw_text)
        predictions.append(pred)
        # print("predictions: ",predictions)

100%|██████████| 15479/15479 [1:41:01<00:00,  2.55it/s]


In [54]:
out_df = df[["title", "body", "priority", "team"]].copy()
out_df["raw_output"] = predictions

out_path = "/content/results/eval_fewshot_base_predictions.csv"
out_df.to_csv(out_path, index=False)

print(f"Saved predictions to {out_path}")

Saved predictions to /content/results/eval_fewshot_base_predictions.csv


In [55]:
"""
metrics.py
----------
Evaluation metrics for the incident triage model.

Parses model output strings of the form:
    severity:P1 | team:platform

Computes:
  - Severity macro-F1   (across P0–P4, all classes weighted equally)
  - Severity accuracy
  - Team routing accuracy
  - Per-class severity F1
  - Confusion matrix data
  - Parse failure rate  (malformed outputs — counted as wrong, not crashed)

Usage — standalone eval on a saved checkpoint:
    python -m src.eval.metrics \
        --checkpoint checkpoints/sft \
        --split val

Usage — imported in trainer for per-epoch eval:
    from src.eval.metrics import evaluate_dataframe
    results = evaluate_dataframe(df, predictions)
"""

import argparse
import os
import re
from typing import Optional

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

# ── constants ─────────────────────────────────────────────────────────────────
PRIORITY_LABELS = ["P0", "P1", "P2", "P3", "P4"]
TEAM_LABELS     = ["platform", "database", "frontend", "backend",
                   "infra", "security", "mobile","unknown"]  # 'unknown' for unparseable team predictions

# ── output parser ─────────────────────────────────────────────────────────────

# Matches: "severity:P1 | team:platform"  (whitespace-tolerant)
_OUTPUT_RE = re.compile(
    r"severity\s*:\s*(P[0-4])"
    r"\s*\|\s*"
    r"team\s*:\s*(\w+)",
    re.IGNORECASE,
)


def parse_output(text: str) -> tuple[Optional[str], Optional[str]]:
    """
    Parse a model output string into (severity, team).

    Returns (None, None) for malformed outputs — caller treats these as wrong
    predictions rather than raising an exception.

    Examples:
        "severity:P1 | team:platform"  → ("P1", "platform")
        "severity:P0|team:infra"       → ("P0", "infra")
        "some garbage text"            → (None, None)
    """
    if not isinstance(text, str):
        return None, None

    # Try to find the pattern anywhere in the output
    # (model sometimes generates extra text before/after)
    match = _OUTPUT_RE.search(text)
    if not match:
        return None, None

    severity = match.group(1).upper()
    team     = match.group(2).lower()

    # Validate values
    if severity not in PRIORITY_LABELS:
        severity = None
    if team not in TEAM_LABELS:
        team = None

    return severity, team


def parse_outputs_batch(texts: list[str]) -> tuple[list[Optional[str]], list[Optional[str]]]:
    """Parse a list of output strings. Returns (severities, teams)."""
    severities, teams = [], []
    for t in texts:
        s, tm = parse_output(t)
        severities.append(s)
        teams.append(tm)
    return severities, teams


# ── metrics ───────────────────────────────────────────────────────────────────

def compute_metrics(
    true_severities: list[str],
    pred_severities: list[Optional[str]],
    true_teams: list[str],
    pred_teams: list[Optional[str]],
    verbose: bool = True,
) -> dict:
    """
    Compute all evaluation metrics.

    Malformed predictions (None) are treated as wrong — they never match the
    true label, so they naturally reduce F1/accuracy.

    Returns a dict with keys:
        severity_macro_f1, severity_accuracy,
        team_accuracy,
        parse_failure_rate,
        per_class_f1  (dict P0..P4 → f1),
        confusion_matrix (list of lists)
    """
    n = len(true_severities)
    assert n == len(pred_severities) == len(true_teams) == len(pred_teams), \
        "All input lists must be the same length."

    # ── parse failure rate ────────────────────────────────────────────────────
    sev_failures  = sum(1 for s in pred_severities if s is None)
    team_failures = sum(1 for t in pred_teams if t is None)
    parse_failure_rate = sev_failures / n

    # ── replace None with a sentinel that never matches ───────────────────────
    SENTINEL_SEV  = "__PARSE_FAIL__"
    SENTINEL_TEAM = "__PARSE_FAIL__"
    pred_sev_clean  = [s if s is not None else SENTINEL_SEV  for s in pred_severities]
    pred_team_clean = [t if t is not None else SENTINEL_TEAM for t in pred_teams]

    # ── severity metrics ──────────────────────────────────────────────────────
    severity_macro_f1 = f1_score(
        true_severities, pred_sev_clean,
        labels=PRIORITY_LABELS, average="macro", zero_division=0
    )
    severity_accuracy = accuracy_score(true_severities, pred_sev_clean)

    per_class_f1 = f1_score(
        true_severities, pred_sev_clean,
        labels=PRIORITY_LABELS, average=None, zero_division=0
    )
    per_class_f1_dict = dict(zip(PRIORITY_LABELS, per_class_f1.tolist()))

    cm = confusion_matrix(true_severities, pred_sev_clean, labels=PRIORITY_LABELS)

    # ── team metrics ──────────────────────────────────────────────────────────
    team_accuracy = accuracy_score(true_teams, pred_team_clean)

    results = {
        "severity_macro_f1"  : round(severity_macro_f1, 4),
        "severity_accuracy"  : round(severity_accuracy, 4),
        "team_accuracy"      : round(team_accuracy, 4),
        "parse_failure_rate" : round(parse_failure_rate, 4),
        "per_class_f1"       : {k: round(v, 4) for k, v in per_class_f1_dict.items()},
        "confusion_matrix"   : cm.tolist(),
        "n_samples"          : n,
    }

    if verbose:
        _print_results(results, true_severities, pred_sev_clean)

    return results


def _print_results(results: dict, true_sev: list, pred_sev: list):
    """Pretty-print evaluation results to stdout."""
    print("=" * 55)
    print("Evaluation Results")
    print("=" * 55)
    print(f"  Samples            : {results['n_samples']:,}")
    print(f"  Parse failure rate : {results['parse_failure_rate']:.1%}")
    print()
    print(f"  Severity macro-F1  : {results['severity_macro_f1']:.4f}  (target > 0.72)")
    print(f"  Severity accuracy  : {results['severity_accuracy']:.4f}")
    print()
    print(f"  Team accuracy      : {results['team_accuracy']:.4f}  (target > 0.78)")
    print()
    print("  Per-class severity F1:")
    for label, f1 in results["per_class_f1"].items():
        bar = "█" * int(f1 * 20)
        print(f"    {label}  {f1:.4f}  {bar}")
    print()
    print("  Severity confusion matrix (rows=true, cols=pred):")
    header = "       " + "  ".join(f"{l:>4}" for l in PRIORITY_LABELS)
    print(header)
    for label, row in zip(PRIORITY_LABELS, results["confusion_matrix"]):
        print(f"  {label}  " + "  ".join(f"{v:>4}" for v in row))
    print()
    print("  Critical off-diagonal cells:")
    cm = results["confusion_matrix"]
    labels = PRIORITY_LABELS
    for true_idx, true_l in enumerate(labels):
        for pred_idx, pred_l in enumerate(labels):
            if true_idx == pred_idx:
                continue
            v = cm[true_idx][pred_idx]
            if v == 0:
                continue
            severity = "🔴 CRITICAL" if (true_l in ("P0","P1") and pred_l in ("P3","P4")) \
                       else "🟡 notable" if true_l in ("P0","P1") \
                       else ""
            if severity:
                print(f"    true={true_l} → pred={pred_l}: {v:,} {severity}")
    print("=" * 55)


# ── DataFrame-level evaluator ─────────────────────────────────────────────────

def evaluate_dataframe(df: pd.DataFrame, predictions: list[str], verbose: bool = True) -> dict:
    """
    Evaluate predictions against a ground-truth DataFrame.

    Args:
        df:          DataFrame with columns 'priority' and 'team'
        predictions: list of raw model output strings, same length as df
        verbose:     print results to stdout

    Returns:
        metrics dict (same structure as compute_metrics)
    """
    pred_sev, pred_team = parse_outputs_batch(predictions)

    return compute_metrics(
        true_severities = df["priority"].tolist(),
        pred_severities = pred_sev,
        true_teams      = df["team"].tolist(),
        pred_teams      = pred_team,
        verbose         = verbose,
    )


# ── standalone inference + eval ───────────────────────────────────────────────

# def run_eval_on_checkpoint(checkpoint_dir: str, split: str = "val"):
#     """
#     Load a saved checkpoint, run inference on the given split, and print metrics.
#     Requires: bitsandbytes (Linux/CUDA only).
#     """
#     import torch
#     from peft import PeftModel
#     from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
#     from tqdm import tqdm

#     ROOT          = os.path.join(os.path.dirname(__file__), "../..")
#     PROCESSED_DIR = os.path.join(ROOT, "data/processed")

#     # ── load data ─────────────────────────────────────────────────────────────
#     df = pd.read_parquet(os.path.join(PROCESSED_DIR, f"{split}.parquet"))
#     print(f"Loaded {split} split: {len(df):,} rows")

#     # ── load model ────────────────────────────────────────────────────────────
#     bnb_config = BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_quant_type="nf4",
#         bnb_4bit_compute_dtype=torch.bfloat16,
#         bnb_4bit_use_double_quant=True,
#     )
#     hf_token  = os.environ.get("HF_TOKEN")
#     base_name = "meta-llama/Llama-3.2-1B"

#     print(f"Loading base model: {base_name}")
#     base_model = AutoModelForCausalLM.from_pretrained(
#         base_name,
#         quantization_config = bnb_config,
#         device_map          = "auto",
#         token               = hf_token,
#         torch_dtype         = torch.bfloat16,
#     )
#     tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)

#     if tokenizer.pad_token is None:
#         tokenizer.pad_token = tokenizer.eos_token
#     tokenizer.truncation_side = "left"
#     print(f"Loading LoRA adapters from: {checkpoint_dir}")
#     model = PeftModel.from_pretrained(base_model, checkpoint_dir)
#     model.eval()

#     # ── inference ─────────────────────────────────────────────────────────────
#     PROMPT_TEMPLATE = "### Incident report:\n{title}\n{body}\n### Triage:\n"
#     predictions = []

#     with torch.inference_mode():
#         for _, row in tqdm(df.iterrows(), total=len(df), desc="Inference"):
#             body   = row["body"].strip() if row["body"] else ""
#             prompt = PROMPT_TEMPLATE.format(
#                 title=row["title"].strip(),
#                 body=("\n" + body) if body else "",
#             )
#             inputs = tokenizer(
#                 prompt, return_tensors="pt", truncation=True, max_length=2048
#             ).to(model.device)

#             output = model.generate(
#                 **inputs,
#                 max_new_tokens  = 20,
#                 do_sample       = False,
#                 pad_token_id    = tokenizer.eos_token_id,
#                 eos_token_id    = tokenizer.eos_token_id,
#             )
#             # Decode only the newly generated tokens
#             generated = output[0][inputs["input_ids"].shape[1]:]
#             predictions.append(tokenizer.decode(generated, skip_special_tokens=True))

#         # ── save all predictions ──────────────────────────────────────────────────
#     os.makedirs(os.path.join(ROOT, "results"), exist_ok=True)
#     pred_sev, pred_team = parse_outputs_batch(predictions)
#     results_df = df[["title", "body", "priority", "team"]].copy()
#     results_df["input_prompt"]    = [
#         PROMPT_TEMPLATE.format(
#             title=row["title"].strip(),
#             body=("\n" + row["body"].strip()) if row["body"] else "",
#         )
#         for _, row in df.iterrows()
#     ]
#     results_df["raw_output"]      = predictions
#     results_df["pred_severity"]   = pred_sev
#     results_df["pred_team"]       = pred_team
#     results_df["sev_correct"]     = results_df["priority"] == results_df["pred_severity"]
#     results_df["team_correct"]    = results_df["team"]     == results_df["pred_team"]
#     results_df.to_csv(os.path.join(ROOT, f"results/eval_{split}_predictions.csv"), index=False)
#     print(f"Saved all predictions to results/eval_{split}_predictions.csv")

#     # ── metrics ───────────────────────────────────────────────────────────────
#     results = evaluate_dataframe(df, predictions, verbose=True)
#     return results


# ── entry point ───────────────────────────────────────────────────────────────

# def main():
#     parser = argparse.ArgumentParser(description="Evaluate a triage checkpoint.")
#     parser.add_argument("--checkpoint", required=True, help="Path to LoRA checkpoint dir.")
#     parser.add_argument("--split", default="val", choices=["val", "test"],
#                         help="Dataset split to evaluate on.")
#     args = parser.parse_args()
#     run_eval_on_checkpoint(args.checkpoint, args.split)




In [56]:
import sys
# sys.path.insert(0, "src")
import pandas as pd
# from eval.metrics import evaluate_dataframe

df = pd.read_csv("/content/results/eval_fewshot_base_predictions.csv")

# evaluate_dataframe expects df with 'priority' and 'team' columns
# and a list of raw output strings
results = evaluate_dataframe(df, df["raw_output"].tolist(), verbose=True)

Evaluation Results
  Samples            : 15,479
  Parse failure rate : 0.0%

  Severity macro-F1  : 0.1032  (target > 0.72)
  Severity accuracy  : 0.0895

  Team accuracy      : 0.2003  (target > 0.78)

  Per-class severity F1:
    P0  0.0564  █
    P1  0.0015  
    P2  0.3008  ██████
    P3  0.0005  
    P4  0.1567  ███

  Severity confusion matrix (rows=true, cols=pred):
         P0    P1    P2    P3    P4
  P0   351     4   218    13    10
  P1   874     1   263    10     7
  P2  1937    21   772    18    14
  P3  7319     2   627     2     8
  P4  1380   181   491   697   259

  Critical off-diagonal cells:
    true=P0 → pred=P1: 4 🟡 notable
    true=P0 → pred=P2: 218 🟡 notable
    true=P0 → pred=P3: 13 🔴 CRITICAL
    true=P0 → pred=P4: 10 🔴 CRITICAL
    true=P1 → pred=P0: 874 🟡 notable
    true=P1 → pred=P2: 263 🟡 notable
    true=P1 → pred=P3: 10 🔴 CRITICAL
    true=P1 → pred=P4: 7 🔴 CRITICAL
